# Experimentacion MobileNetV2 - TFLite FP16

Notebook para cargar el modelo entrenado `mobilenetv2_v3_fp16.tflite`, clasificar imagenes subidas desde el computador y registrar las metricas del modelo.

Modelo evaluado: `mobilenetv2_v3_fp16.tflite`  
Dataset base: PlantVillage, 38 clases  
Hoja de registro: https://docs.google.com/spreadsheets/d/13_oqj5JWWKHnTzcsgkbcmKLMdpNZmSif45y6sf3gGeo/edit?gid=1383304375

**Metricas a registrar:**
- Tamano del archivo `.tflite` (MB)
- Tiempo de inferencia promedio (ms)
- Clasificacion obtenida por imagen
- Confianza del modelo

In [4]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'ai-edge-litert', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('ai-edge-litert instalado correctamente.')
else:
    print('No se pudo instalar ai-edge-litert, se usara tf.lite como fallback.')
    print(result.stderr[:300])

ai-edge-litert instalado correctamente.


In [5]:
from pathlib import Path
import time
import os
import warnings

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

# Usar ai_edge_litert si esta disponible, fallback a tf.lite suprimiendo el warning
try:
    from ai_edge_litert.interpreter import Interpreter as LiteInterpreter
    LITERT_BACKEND = 'ai_edge_litert'
except ImportError:
    warnings.filterwarnings('ignore', message='.*tf.lite.Interpreter is deprecated.*')
    LiteInterpreter = tf.lite.Interpreter
    LITERT_BACKEND = 'tf.lite (fallback)'

print('TensorFlow:', tf.__version__)
print('Interprete TFLite:', LITERT_BACKEND)

TensorFlow: 2.20.0
Interprete TFLite: ai_edge_litert


## Configuracion de rutas

In [6]:
BASE_DIR = Path('/Users/carolinachavez/convolutional-neuronal-network/MobileNet')

TFLITE_MODEL_PATH = BASE_DIR / 'saved_models' / 'mobilenetv2_v3' / 'mobilenetv2_v3_fp16.tflite'
LABELS_PATH = BASE_DIR / 'saved_models' / 'mobilenetv2_v3' / 'labels.txt'

IMG_SIZE = (224, 224)
NUM_INFERENCE_RUNS = 10  # repeticiones para medir latencia promedio

assert TFLITE_MODEL_PATH.exists(), f'Modelo no encontrado: {TFLITE_MODEL_PATH}'
assert LABELS_PATH.exists(), f'Labels no encontradas: {LABELS_PATH}'

print('Modelo:', TFLITE_MODEL_PATH)
print('Labels:', LABELS_PATH)

Modelo: /Users/carolinachavez/convolutional-neuronal-network/MobileNet/saved_models/mobilenetv2_v3/mobilenetv2_v3_fp16.tflite
Labels: /Users/carolinachavez/convolutional-neuronal-network/MobileNet/saved_models/mobilenetv2_v3/labels.txt


## Carga del modelo y etiquetas

In [7]:
# Tamano del modelo en MB
model_size_mb = TFLITE_MODEL_PATH.stat().st_size / (1024 ** 2)
print(f'Tamano del modelo TFLite FP16: {model_size_mb:.4f} MB')

# Cargar el interprete TFLite
interpreter = LiteInterpreter(model_path=str(TFLITE_MODEL_PATH))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('\nInput details:')
print('  Shape:', input_details[0]['shape'])
print('  Dtype:', input_details[0]['dtype'])
print('Output details:')
print('  Shape:', output_details[0]['shape'])

# Cargar etiquetas
with open(LABELS_PATH, 'r') as f:
    labels = [line.strip() for line in f.readlines()]

print(f'\nClases cargadas: {len(labels)}')

Tamano del modelo TFLite FP16: 4.3494 MB

Input details:
  Shape: [  1 224 224   3]
  Dtype: <class 'numpy.float32'>
Output details:
  Shape: [ 1 38]

Clases cargadas: 38


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## Funciones de preprocesamiento e inferencia

In [8]:
def preprocess_image(img_path: str) -> np.ndarray:
    """Carga y preprocesa una imagen para inferencia con MobileNetV3."""
    img = Image.open(img_path).convert('RGB')
    img = img.resize(IMG_SIZE)
    img_array = np.array(img, dtype=np.float32)
    # Normalizacion a [-1, 1] alineada con el entrenamiento y la app Android
    img_array = (img_array - 127.5) / 127.5
    img_array = np.expand_dims(img_array, axis=0)
    return img_array


def run_inference(img_array: np.ndarray, n_runs: int = NUM_INFERENCE_RUNS):
    """Ejecuta inferencia n veces y retorna predicciones + latencia promedio en ms."""
    latencies = []
    predictions = None
    for _ in range(n_runs):
        interpreter.set_tensor(input_details[0]['index'], img_array)
        t0 = time.perf_counter()
        interpreter.invoke()
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)
        predictions = interpreter.get_tensor(output_details[0]['index'])
    avg_latency_ms = np.mean(latencies)
    return predictions[0], avg_latency_ms


def top_k_results(predictions: np.ndarray, k: int = 5):
    """Retorna las top-k clases con sus probabilidades."""
    top_indices = np.argsort(predictions)[::-1][:k]
    return [(labels[i], float(predictions[i])) for i in top_indices]


print('Funciones definidas correctamente.')

Funciones definidas correctamente.


## Medicion de latencia base (imagen sintetica)

Se ejecuta inferencia sobre una imagen aleatoria para obtener la latencia promedio del modelo antes de procesar imagenes reales.

In [9]:
dummy_input = np.random.uniform(-1, 1, (1, 224, 224, 3)).astype(np.float32)
_, avg_latency_ms = run_inference(dummy_input, n_runs=20)

print('=== Metricas del modelo (para registrar en el Excel) ===')
print(f'Modelo:                  mobilenetv2_v3_fp16.tflite')
print(f'Tamano (MB):             {model_size_mb:.4f}')
print(f'Latencia promedio (ms):  {avg_latency_ms:.2f}  (promedio de 20 ejecuciones)')
print(f'Tipo de cuantizacion:    FP16')
print(f'Clases:                  {len(labels)}')

=== Metricas del modelo (para registrar en el Excel) ===
Modelo:                  mobilenetv2_v3_fp16.tflite
Tamano (MB):             4.3494
Latencia promedio (ms):  10.05  (promedio de 20 ejecuciones)
Tipo de cuantizacion:    FP16
Clases:                  38


## Clasificacion de imagenes desde el computador

Sube una o varias imagenes usando el widget. El modelo mostrara la clase predicha, la confianza y el tiempo de inferencia.

In [10]:
import io

upload_widget = widgets.FileUpload(
    accept='image/*',
    multiple=True,
    description='Subir imagenes',
    layout=widgets.Layout(width='300px')
)

run_button = widgets.Button(
    description='Clasificar',
    button_style='primary',
    layout=widgets.Layout(width='150px')
)

output_area = widgets.Output()

results_log = []

def on_classify(b):
    with output_area:
        clear_output(wait=True)
        if not upload_widget.value:
            print('Sube al menos una imagen primero.')
            return

        # ipywidgets >= 8.0: value es una tupla de dicts con claves 'name' y 'content'
        for file_info in upload_widget.value:
            filename = file_info['name']
            content = file_info['content']  # bytes
            img = Image.open(io.BytesIO(content)).convert('RGB')
            img_resized = img.resize(IMG_SIZE)
            img_array = np.array(img_resized, dtype=np.float32)
            img_array = (img_array - 127.5) / 127.5
            img_array = np.expand_dims(img_array, axis=0)

            predictions, latency_ms = run_inference(img_array)
            top5 = top_k_results(predictions, k=5)
            top1_class, top1_conf = top5[0]

            results_log.append({
                'archivo': filename,
                'prediccion': top1_class,
                'confianza': round(top1_conf * 100, 2),
                'latencia_ms': round(latency_ms, 2)
            })

            fig, axes = plt.subplots(1, 2, figsize=(12, 4))

            axes[0].imshow(img)
            axes[0].axis('off')
            axes[0].set_title(f'{filename}', fontsize=10)

            classes = [c for c, _ in top5]
            confidences = [conf * 100 for _, conf in top5]
            colors = ['#2196F3' if i > 0 else '#4CAF50' for i in range(len(classes))]
            bars = axes[1].barh(classes[::-1], confidences[::-1], color=colors[::-1])
            axes[1].set_xlabel('Confianza (%)')
            axes[1].set_title('Top-5 predicciones')
            axes[1].set_xlim(0, 100)
            for bar, conf in zip(bars, confidences[::-1]):
                axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                             f'{conf:.1f}%', va='center', fontsize=9)

            plt.tight_layout()
            plt.show()

            print(f'Archivo:     {filename}')
            print(f'Prediccion:  {top1_class}')
            print(f'Confianza:   {top1_conf * 100:.2f}%')
            print(f'Latencia:    {latency_ms:.2f} ms')
            print('-' * 50)

run_button.on_click(on_classify)
display(widgets.VBox([upload_widget, run_button, output_area]))

## Resumen de resultados para registrar en el Excel

Ejecuta esta celda despues de clasificar imagenes para ver el resumen consolidado.

In [14]:
import pandas as pd

if results_log:
    df = pd.DataFrame(results_log)

    print('=== Resumen de clasificaciones ===')
    display(df)

    print('\n=== Metricas globales del modelo ===')
    print(f'Modelo:                  mobilenetv2_v3_fp16.tflite')
    print(f'Tamano (MB):             {model_size_mb:.4f}')
    print(f'Latencia promedio (ms):  {df["latencia_ms"].mean():.2f}')
    print(f'Confianza promedio (%):  {df["confianza"].mean():.2f}')
    print(f'Imagenes evaluadas:      {len(df)}')
    print(f'Tipo cuantizacion:       FP16')
    print(f'Clases del modelo:       {len(labels)}')

    print('\n=== Fila lista para Excel ===' )
    print(f'mobilenetv2_v3 | FP16 | {model_size_mb:.4f} MB | {df["latencia_ms"].mean():.2f} ms | {len(labels)} clases')
else:
    print('Todavia no se han clasificado imagenes. Usa el widget de arriba.')

=== Resumen de clasificaciones ===


,archivo,prediccion,confianza,latencia_ms
0,1.jpg,Corn___Cercospora_leaf_spot Gray_leaf_spot,91.63,10.41
1,5.JPG,Orange___Haunglongbing_(Citrus_greening),73.34,10.75
2,10.jpg,Strawberry___Leaf_scorch,99.85,9.62
3,11.JPG,Corn___Common_rust,69.68,10.94
4,12.jpg,Corn___Cercospora_leaf_spot Gray_leaf_spot,72.75,10.59
5,17.JPG,Corn___Common_rust,95.05,10.14
6,19.jpg,Strawberry___Leaf_scorch,73.81,9.57
7,21.jpg,Tomato___Late_blight,99.92,9.24
8,23.jpg,Strawberry___Leaf_scorch,99.98,9.42
9,26.jpg,"Pepper,_bell___Bacterial_spot",74.42,10.42



=== Metricas globales del modelo ===
Modelo:                  mobilenetv2_v3_fp16.tflite
Tamano (MB):             4.3494
Latencia promedio (ms):  10.53
Confianza promedio (%):  79.20
Imagenes evaluadas:      16
Tipo cuantizacion:       FP16
Clases del modelo:       38

=== Fila lista para Excel ===
mobilenetv2_v3 | FP16 | 4.3494 MB | 10.53 ms | 38 clases


## Inferencia desde ruta de archivo local

Alternativa al widget: especifica la ruta de una imagen directamente.

In [ ]:
# Cambia esta ruta a la imagen que quieras evaluar
IMAGE_PATH = '/ruta/a/tu/imagen.jpg'

if Path(IMAGE_PATH).exists():
    img_array = preprocess_image(IMAGE_PATH)
    predictions, latency_ms = run_inference(img_array)
    top5 = top_k_results(predictions, k=5)

    img = Image.open(IMAGE_PATH)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].imshow(img)
    axes[0].axis('off')
    axes[0].set_title(Path(IMAGE_PATH).name)

    classes = [c for c, _ in top5]
    confidences = [conf * 100 for _, conf in top5]
    colors = ['#2196F3' if i > 0 else '#4CAF50' for i in range(len(classes))]
    axes[1].barh(classes[::-1], confidences[::-1], color=colors[::-1])
    axes[1].set_xlabel('Confianza (%)')
    axes[1].set_title('Top-5 predicciones')
    axes[1].set_xlim(0, 100)

    plt.tight_layout()
    plt.show()

    print(f'Prediccion:  {top5[0][0]}')
    print(f'Confianza:   {top5[0][1] * 100:.2f}%')
    print(f'Latencia:    {latency_ms:.2f} ms')
else:
    print(f'Archivo no encontrado: {IMAGE_PATH}')
    print('Modifica la variable IMAGE_PATH con la ruta correcta.')